<a href="https://colab.research.google.com/github/PadmavathiMuthukumar/diabetes-prediction-ml-model-/blob/main/diabetes_prediction_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Logistic Regression Model

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

### 1. Load the Dataset

In [2]:
df = pd.read_csv('/content/diabetes.csv')
display(df.head())

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


### 2. Prepare Data (Features and Target)

In [3]:
# Assuming 'Outcome' is the target variable, which is common for diabetes datasets
X = df.drop('Outcome', axis=1)
y = df['Outcome']

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")

Features (X) shape: (768, 8)
Target (y) shape: (768,)


### 3. Split Data into Training and Testing Sets

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (614, 8)
X_test shape: (154, 8)
y_train shape: (614,)
y_test shape: (154,)


### 3.5 Scale Data and Export Scaler

In [11]:
from sklearn.preprocessing import StandardScaler
import joblib

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform both training and test data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Export the fitted scaler
scaler_filename = 'scaler.joblib'
joblib.dump(scaler, scaler_filename)

print(f"Data scaled and scaler successfully exported to '{scaler_filename}'")

# Display the shape of scaled data
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")

Data scaled and scaler successfully exported to 'scaler.joblib'
X_train_scaled shape: (614, 8)
X_test_scaled shape: (154, 8)


### 4. Train the Logistic Regression Model

In [12]:
# Initialize the Logistic Regression model
model = LogisticRegression(solver='liblinear', random_state=42)

# Train the model using scaled training data
model.fit(X_train_scaled, y_train)

print("Logistic Regression model trained successfully on scaled data.")

Logistic Regression model trained successfully on scaled data.


### 5. Evaluate the Model

In [13]:
# Make predictions on the scaled test set
y_pred = model.predict(X_test_scaled)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")

# Display classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 0.7143

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.82      0.79       100
           1       0.61      0.52      0.56        54

    accuracy                           0.71       154
   macro avg       0.68      0.67      0.67       154
weighted avg       0.71      0.71      0.71       154



### 6. Export the Trained Model

In [7]:
import joblib

# Define the filename for the exported model
model_filename = 'logistic_regression_model.joblib'

# Save the trained model to the file
joblib.dump(model, model_filename)

print(f"Model successfully exported to '{model_filename}'")

Model successfully exported to 'logistic_regression_model.joblib'


### How to Load the Exported Model (Optional)

In [8]:
# To load the model back into your environment later:
# loaded_model = joblib.load(model_filename)
# print("Model loaded successfully!")

# You can then use the loaded_model for predictions
# new_predictions = loaded_model.predict(X_new)
# print("Predictions from loaded model:", new_predictions)

## Create a Flask API for the Model (Regenerated)

In [14]:
import joblib
import pandas as pd
from flask import Flask, request, jsonify

# Load the trained model and the scaler
model = joblib.load('logistic_regression_model.joblib')
scaler = joblib.load('scaler.joblib')

# Initialize the Flask application
app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    try:
        # Get JSON data from the request
        data = request.get_json(force=True)

        # Convert dictionary to DataFrame, ensuring correct column order
        feature_names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
        input_df = pd.DataFrame([data], columns=feature_names)

        # Scale the input data using the loaded scaler
        input_scaled = scaler.transform(input_df)

        # Make prediction
        prediction = model.predict(input_scaled)
        probability = model.predict_proba(input_scaled)[:, 1]

        # Return the prediction and probability
        return jsonify({
            'prediction': int(prediction[0]),
            'probability_of_diabetes': float(probability[0])
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 400

@app.route('/', methods=['GET'])
def home():
    return "Welcome to the Diabetes Prediction API! Send POST requests to /predict with your data."

# This part is typically uncommented and run when deploying the Flask app outside of a notebook
# if __name__ == '__main__':
#     app.run(debug=True, host='0.0.0.0', port=5000)

To use this Flask API (you cannot run the Flask server directly in this Colab cell for external access):

1.  **Save the code:** Copy the Python code from the cell above and save it as a file named `app.py` in the same directory where your `logistic_regression_model.joblib` file is located.
2.  **Install Flask:** If you don't have Flask installed, open your terminal or command prompt and run:
    ```bash
    pip install Flask
    ```
3.  **Run the application (locally):** Open your terminal or command prompt, navigate to the directory where you saved `app.py`, and run:
    ```bash
    python app.py
    ```
    You should see output indicating that the Flask development server is running, usually on `http://127.0.0.1:5000/`.

4.  **Send requests:** You can now send POST requests to `http://127.0.0.1:5000/predict` with your input data in JSON format. For example, using `curl`:
    ```bash
    curl -X POST -H "Content-Type: application/json" -d '{"Pregnancies":6,"Glucose":148,"BloodPressure":72,"SkinThickness":35,"Insulin":0,"BMI":33.6,"DiabetesPedigreeFunction":0.627,"Age":50}' http://127.0.0.1:5000/predict
    ```
    Or using a Python script:
    ```python
    import requests
    import json

    url = 'http://127.0.0.1:5000/predict'
    headers = {'Content-Type': 'application/json'}
    data = {
        "Pregnancies": 6,
        "Glucose": 148,
        "BloodPressure": 72,
        "SkinThickness": 35,
        "Insulin": 0,
        "BMI": 33.6,
        "DiabetesPedigreeFunction": 0.627,
        "Age": 50
    }

    response = requests.post(url, headers=headers, data=json.dumps(data))
    print(response.json())
    ```

Alternatively, if you want to test this within **Google Colab** and make it accessible via a public URL, you'll need to set up `ngrok`.

## Create a Flask API for the Model